In [1]:
import os
import json
import numpy as np
from pymilvus import MilvusClient
from pymilvus.model.dense.sentence_transformer import SentenceTransformerEmbeddingFunction
from datasets import load_dataset
from ibm_watsonx_ai.foundation_models import ModelInference
from ibm_watsonx_ai import Credentials
from dotenv import load_dotenv
from tqdm import tqdm
import sys

In [2]:
load_dotenv()

True

In [3]:
# model_path = '../models/google-bert_bert-base-uncased_fmsr_ft_bs32_poolmean_pdesc0.0_positiveratio_0.5seed_42'
model_path = '../models/sentence-transformers_all-mpnet-base-v2_fmsr_ft_bs32_poolmean_pdesc0.4_positiveratio_0.5seed_42'
model_path += '/checkpoint-9600'
if sys.argv[1] != '-f':
    model_path = sys.argv[1]
    model_out_name = sys.argv[2]
if 'bert' in model_path.lower():
    emb_dim = 768
elif 'mpnet' in model_path.lower():
    emb_dim = 768
elif 'baai' in model_path.lower():
    emb_dim = 1024

In [4]:
model_id = 'mistralai/mistral-large'
params = {
    "temperature": 0.0,
    "max_tokens": 2000,
}

credentials = Credentials(
    url=os.environ['WATSONX_URL'],
    api_key=os.environ['WATSONX_APIKEY']
)

llm = ModelInference(
    model_id=model_id,
    credentials=credentials,
    project_id=os.environ['WATSONX_PROJECT_ID'],
    params=params
)
json_format = '''```json
{"reasoning": "<your reasoning>", "answer": ["<answer letter>"]}
```'''

In [5]:
model = SentenceTransformerEmbeddingFunction(model_path)

In [6]:
all_summaries = []
total_w = 0
total_arxiv = 0
second_split = ['Title:', 'Summary:']
for fname in os.listdir('../training/pretrain/all_jsons'):
    if not fname.endswith('.json'):
        continue
    with open('../training/pretrain/all_jsons/' + fname, 'r') as f:
        data = json.load(f)
    sources = [key.split('::')[0].lower() for key in data]
    w_cnt, arxiv_cnt = sources.count('wikipedia'), sources.count('arxiv')
    total_w += w_cnt
    total_arxiv += arxiv_cnt
    for item in data:
        text = data[item]
        for i, keyword in enumerate(['Published:', 'Page:']):
            if keyword in text:
                break
        for txt in text.split(keyword):
            txt = txt.strip()
            if txt == '':
                continue
            try:
                summary = txt.split(second_split[i])[1].strip()
                if summary == '':
                    continue
            except:
                continue
            all_summaries.append(summary)
all_summaries = np.unique(all_summaries)
total_w, total_arxiv

(10552, 11515)

In [8]:
client = MilvusClient(f"./{model_out_name}.db")

/u/modelfactory/.conda/envs/lab/lib/python3.10/site-packages/milvus_lite/__init__.py:15: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [9]:
if client.has_collection(collection_name="industry"):
    client.drop_collection(collection_name="industry")
client.create_collection(
    collection_name="industry",
    dimension=emb_dim, 
)

In [ ]:
print('creating doc embeddings')
vectors = model.encode_documents(all_summaries)
print('created doc embeddings')

In [ ]:
data = [
    {"id": i, "vector": vectors[i], "text": all_summaries[i]}
    for i in range(len(vectors))
]

In [ ]:
res = client.insert(collection_name="industry", data=data)

In [34]:
ds = load_dataset('cc4718/FailureSensorIQ', data_files='all.jsonl')['train']

In [35]:
print('embedding all questions')
q_vecs = model.encode_documents(ds['question'])
print('embedded all questions')

In [ ]:
n_correct = 0
n_invalid = 0
for i, item in enumerate(tqdm(ds)):
    q_vec = [q_vecs[i]]
    search_res = client.search(
        collection_name="industry",
        data=q_vec,
        limit=3,
        output_fields=["text"],
    )
    retrieved = [res["entity"]["text"] for res in search_res[0]]
    prompt = ''
    for doc in retrieved:
        prompt += f'Document: {doc}\n'
    prompt += f"Question:\n{item['question']}"
    for c, opt in zip(item['option_ids'], item['options']):
        prompt += f'{c}. {opt}\n'
    messages = [
        {
            "role": "user", 
            "content": prompt + f'\n{json_format}'
        }
    ]
    while True:
        try:
            response = llm.chat(messages)
            break
        except:
            pass
    raw = response['choices'][0]['message']['content']
    start_idx = raw.find('{')
    end_idx = raw.find('}') + 1
    try:
        cleaned = json.loads(raw[start_idx:end_idx])
        pred_letter = cleaned['answer'][0]
        ans = ord(pred_letter) - ord('A')
        n_correct += int(ans == item['correct'].index(True))    
    except:
        n_invalid += 1
acc = round(n_correct / (len(ds) - n_invalid) * 100, 2)
print(f'acc: {acc}')
invalid_pct = round(n_invalid / len(ds) * 100, 2)
print(f'invalid pct: {invalid_pct}')

  0%|                                                                                                                                                                                        | 0/2667 [00:00<?, ?it/s]

b
a


  0%|                                                                                                                                                                              | 1/2667 [00:02<1:40:04,  2.25s/it]

b
a


  0%|▏                                                                                                                                                                             | 2/2667 [00:04<1:36:45,  2.18s/it]

...

b
a


  1%|██▍                                                                                                                                                                          | 37/2667 [01:13<1:21:08,  1.85s/it]

b
a


  1%|██▍                                                                                                                                                                          | 38/2667 [01:15<1:16:33,  1.75s/it]

b
a
